In [1]:
import random
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from PIL import Image
from sklearn.metrics import auc, average_precision_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation, Resize, ToTensor
from tqdm import tqdm

In [2]:
SEED = 492
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

In [3]:
data_path = Path("../../data/ISIC")
image_path = data_path / "ISIC_2024_Training_Input"
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"
metadata_path = data_path / "metadata.csv"

model_path = Path("../../models")

In [4]:
class ISICDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, transform):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        img = Image.open(row["image_path"]).convert("RGB")
        return self.transform(img), torch.tensor(row["malignant"], dtype=torch.float32)

In [5]:
train_transform = Compose([Resize((224, 224)), RandomHorizontalFlip(p=0.5), RandomRotation(15), ToTensor()])

val_transform = Compose([Resize((224, 224)), ToTensor()])

In [6]:
ground_truth_df = pl.read_csv(ground_truth_path)
metadata_df = pl.read_csv(metadata_path)

df = ground_truth_df.join(metadata_df, on="isic_id", how="inner").with_columns(
    (pl.lit(str(image_path)) + "/" + pl.col("isic_id").cast(pl.Utf8) + ".jpg").alias("image_path")
)

patient_stats = df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_malignant")).sort("patient_id")

train_p, val_p = train_test_split(
    patient_stats, test_size=0.1, random_state=SEED, stratify=patient_stats["has_malignant"]
)

train_ids = train_p["patient_id"].to_list()
val_ids = val_p["patient_id"].to_list()

train_df = df.filter(pl.col("patient_id").is_in(train_ids))
val_df = df.filter(pl.col("patient_id").is_in(val_ids))

train_malignant = train_df.filter(pl.col("malignant") == 1)
train_benign = train_df.filter(pl.col("malignant") == 0)

capped_benign = (
    train_benign.sample(fraction=1.0, shuffle=True, seed=SEED)
    .group_by("patient_id")
    .head(20)
    .select(train_df.columns)
)

final_train_df = pl.concat([train_malignant, capped_benign])

train_loader = DataLoader(ISICDataset(final_train_df, train_transform), batch_size=128, shuffle=True, generator=g)
val_loader = DataLoader(ISICDataset(val_df, val_transform), batch_size=128, shuffle=False)

In [7]:
train_malignant_count = final_train_df.filter(pl.col("malignant") == 1).height
train_benign_count = final_train_df.filter(pl.col("malignant") == 0).height
val_malignant_count = val_df.filter(pl.col("malignant") == 1).height
val_benign_count = val_df.filter(pl.col("malignant") == 0).height

print(
    f"Train - Malignant: {train_malignant_count}, "
    f"Benign: {train_benign_count}, "
    f"Ratio: 1:{train_benign_count / train_malignant_count:.1f}"
)
print(
    f"Val - Malignant: {val_malignant_count}, "
    f"Benign: {val_benign_count}, "
    f"Ratio: 1:{val_benign_count / val_malignant_count:.1f}"
)

Train - Malignant: 348, Benign: 18392, Ratio: 1:52.9
Val - Malignant: 45, Benign: 47988, Ratio: 1:1066.4


In [8]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, 1))
device = torch.device("cuda")
model = model.to(device)

In [9]:
pos_weight = torch.tensor([train_benign_count / train_malignant_count]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=3e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=3)
scaler = torch.amp.GradScaler("cuda")

In [10]:
best_model_state = None
best_recall_at_spec = 0.0
best_ap = 0.0
early_stopping_counter = 0
early_stopping_patience = 5

In [11]:
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", leave=False)
    for inputs, labels in train_loop:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(inputs)
            loss = criterion(outputs, labels.unsqueeze(1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", leave=False)

    model_path.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(inputs)
                loss = criterion(outputs, labels.unsqueeze(1))
            val_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    ap = average_precision_score(all_labels, all_preds)
    fpr, tpr, thresholds = roc_curve(all_labels, all_preds)

    valid_indices = np.where(fpr <= 0.10)[0]
    recall_at_spec = tpr[valid_indices[-1]]

    scheduler.step(ap)

    if ap > best_ap:
        best_ap = ap
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if recall_at_spec > best_recall_at_spec:
        best_recall_at_spec = recall_at_spec
        torch.save(model.state_dict(), model_path / "best_image_model.pt")

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
        f"AP: {ap:.4f} | Recall@90Spec: {recall_at_spec:.4f}"
    )

    if early_stopping_counter >= early_stopping_patience:
        print(f"Early stopping triggered at epoch {epoch + 1}")
        break

Epoch 1/30 | Train Loss: 1.1736 | Val Loss: 0.6252 | AP: 0.0146 | Recall@90Spec: 0.6889


Epoch 2/30 | Train Loss: 0.7948 | Val Loss: 0.5015 | AP: 0.0220 | Recall@90Spec: 0.7556


Epoch 3/30 | Train Loss: 0.6472 | Val Loss: 0.3454 | AP: 0.0472 | Recall@90Spec: 0.6889


Epoch 4/30 | Train Loss: 0.5228 | Val Loss: 0.2990 | AP: 0.0349 | Recall@90Spec: 0.6889


Epoch 5/30 | Train Loss: 0.4446 | Val Loss: 0.2630 | AP: 0.0555 | Recall@90Spec: 0.7111


Epoch 6/30 | Train Loss: 0.3359 | Val Loss: 0.2422 | AP: 0.0258 | Recall@90Spec: 0.7111


Epoch 7/30 | Train Loss: 0.2613 | Val Loss: 0.2536 | AP: 0.0160 | Recall@90Spec: 0.6444


Epoch 8/30 | Train Loss: 0.2880 | Val Loss: 0.2277 | AP: 0.0384 | Recall@90Spec: 0.7111


Epoch 9/30 | Train Loss: 0.2204 | Val Loss: 0.2113 | AP: 0.0462 | Recall@90Spec: 0.7556


Epoch 10/30 | Train Loss: 0.1918 | Val Loss: 0.1997 | AP: 0.0477 | Recall@90Spec: 0.7556
Early stopping triggered at epoch 10


In [12]:
model.load_state_dict(torch.load(model_path / "best_image_model.pt", weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Evaluating Best Model"):
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(inputs)
        all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

auc_roc = roc_auc_score(all_labels, all_preds)
ap = average_precision_score(all_labels, all_preds)

fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
valid_idx = np.where(fpr <= 0.10)[0]
threshold_90_spec = thresholds[valid_idx[-1]]
recall_90_spec = tpr[valid_idx[-1]]

preds_binary = (all_preds >= threshold_90_spec).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds_binary).ravel()

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"Average Precision: {ap:.4f}")
print(f"Threshold @ 90% Spec: {threshold_90_spec:.4f}")
print(f"Recall @ 90% Spec: {recall_90_spec:.4f}")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")

Evaluating Best Model: 100%|██████████| 376/376 [01:18<00:00,  4.78it/s]

AUC-ROC: 0.8971
Average Precision: 0.0220
Threshold @ 90% Spec: 0.6367
Recall @ 90% Spec: 0.7556
TP: 34 | FP: 4789 | TN: 43199 | FN: 11


In [13]:
def p_auc_tpr(v_gt, v_pred, min_tpr=0.80):
    v_gt_flipped = abs(np.asarray(v_gt) - 1)
    v_pred_flipped = abs(np.asarray(v_pred) - 1)
    max_fpr = abs(1 - min_tpr)

    fpr, tpr, _ = roc_curve(v_gt_flipped, v_pred_flipped)

    stop = np.searchsorted(fpr, max_fpr, "right")
    x_interp = [fpr[stop - 1], fpr[stop]]
    y_interp = [tpr[stop - 1], tpr[stop]]

    tpr_adj = np.append(tpr[:stop], np.interp(max_fpr, x_interp, y_interp))
    fpr_adj = np.append(fpr[:stop], max_fpr)

    return auc(fpr_adj, tpr_adj)


isic_pauc = p_auc_tpr(all_labels, all_preds, min_tpr=0.80)
print(f"ISIC 2024 Official Metric (pAUC > 80% TPR): {isic_pauc:.5f}")

ISIC 2024 Official Metric (pAUC > 80% TPR): 0.12417
